# Product Comparison Pipeline

**Task owner:** Fatemeh  
**Current scope:** Product comparison using product facts only

This notebook implements the current product-comparison pipeline without any LLM or external API request.

## Current pipeline

```text
Product queries
    ↓
Rule-based query handling
(adapted from Benyamin)
    ↓
Hybrid product retrieval
(adapted from Ali)
    ↓
Real product artifacts from shared Google Drive
    ↓
Direct product facts
    ↓
Structured product comparison

Review evidence  → unavailable until the real comment index is published
Inference        → disabled
LLM/API calls    → none
```

## Source attribution

- Persian normalization: adapted from Ali's `src/data/normalize.py`
- Product retrieval contract and product retrievers: adapted from Ali's `src/retrieval/base.py`, `src/retrieval/products.py`, and `src/retrieval/hybrid.py`
- Rule-based filter extraction and product-discovery pattern: adapted from Benyamin's `src/chains/product_filters.py` and `src/chains/product_discovery.py`
- Facts/evidence/inference separation and the comparison layer: implemented for the Product Comparison task

This notebook does not rebuild BM25, embeddings, or FAISS indexes. It only loads the shared retrieval artifacts and queries them.

## 1. Mount Google Drive

The retrieval artifacts are stored in the shared Google Drive.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Configuration

Hybrid retrieval is the default backend because it achieved the best retrieval score in Ali's evaluation:

```text
BM25 only     nDCG@10 = 0.6389
Dense only    nDCG@10 = 0.7329
Hybrid        nDCG@10 = 0.7778
```

The hybrid retriever combines lexical BM25 retrieval and semantic dense retrieval using Reciprocal Rank Fusion (RRF).

Expected artifacts:

```text
products_meta_v1.parquet
products_bm25_v1.npz
products_bm25_vocab_v1.json
products_e5base_ivfsq8_v1.faiss
```

In [2]:
from pathlib import Path

INDEX_DIR = Path("/content/drive/MyDrive/DigikalaProject/indexes")

BACKEND = "hybrid"
INDEX_TYPE = "ivfsq8"

print("INDEX_DIR:", INDEX_DIR)
print("BACKEND:", BACKEND)
print("INDEX_TYPE:", INDEX_TYPE)

INDEX_DIR: /content/drive/MyDrive/DigikalaProject/indexes
BACKEND: hybrid
INDEX_TYPE: ivfsq8


## 3. Runtime dependencies

Hybrid retrieval requires:

- `pandas` and `pyarrow` for product metadata
- `scipy` for the BM25 sparse matrix
- `faiss-cpu` for vector search
- `sentence-transformers` for query embeddings

A GPU is optional. The dense encoder automatically uses CUDA when available and otherwise runs on CPU.

In [3]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or import_name]
        )

ensure_package("pyarrow")
ensure_package("scipy")
ensure_package("faiss", "faiss-cpu")
ensure_package("sentence_transformers", "sentence-transformers")

print("Runtime dependencies are available.")

Runtime dependencies are available.


## 4. Artifact validation

This cell verifies that the shared product-retrieval artifacts exist before loading the indexes.

In [4]:
required_files = [
    "products_meta_v1.parquet",
    "products_bm25_v1.npz",
    "products_bm25_vocab_v1.json",
    f"products_e5base_{INDEX_TYPE}_v1.faiss",
]

missing_files = [
    file_name
    for file_name in required_files
    if not (INDEX_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing retrieval artifacts in INDEX_DIR:\n- "
        + "\n- ".join(missing_files)
    )

print("All required retrieval artifacts were found.")

All required retrieval artifacts were found.


## 5. Shared Persian text normalization

This section is adapted from Ali's shared normalizer.

The same normalization logic is used across retrieval so that Persian/Arabic character variants, Persian/Arabic digits, diacritics, invisible Unicode controls, repeated characters, and inconsistent spacing do not create avoidable search mismatches.

In [5]:
# Adapted from src/data/normalize.py — Owner: Ali

import re
import unicodedata

ZWNJ = "\u200c"

_CHAR_MAP: dict[str, str] = {
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "أ": "ا",
    "إ": "ا",
    "ٱ": "ا",
    "ة": "ه",
    "ۀ": "ه",
    "ؤ": "و",
    "٫": ".",
    "٬": ",",
    "،": ",",
    "؛": ";",
    "؟": "?",
    "٪": "%",
    "«": '"',
    "»": '"',
}

for _i, (_fa, _ar) in enumerate(zip("۰۱۲۳۴۵۶۷۸۹", "٠١٢٣٤٥٦٧٨٩")):
    _CHAR_MAP[_fa] = str(_i)
    _CHAR_MAP[_ar] = str(_i)

_DROP_CHARS = "".join(
    chr(c)
    for c in [
        *range(0x064B, 0x0653),
        0x0654,
        0x0655,
        0x0670,
        0x0640,
        0x200B,
        0x200D,
        0x200E,
        0x200F,
        0x202A,
        0x202B,
        0x202C,
        0x202D,
        0x202E,
        0x2066,
        0x2067,
        0x2068,
        0x2069,
        0xFEFF,
    ]
)

_SPACE_CHARS = "".join(
    chr(c)
    for c in [0x00A0, *range(0x2000, 0x200B), 0x202F, 0x205F, 0x3000]
)

_TRANSLATE: dict[int, str | None] = {ord(k): v for k, v in _CHAR_MAP.items()}
_TRANSLATE.update({ord(c): None for c in _DROP_CHARS})
_TRANSLATE.update({ord(c): " " for c in _SPACE_CHARS})

_EMOJI_RE = re.compile(
    "["
    "\U0001f000-\U0001faff"
    "\u2190-\u21ff"
    "\u2600-\u27bf"
    "\u2b00-\u2bff"
    "\ufe00-\ufe0f"
    "\u2049\u203c\u2122"
    "]+"
)
_REPEAT_FA_RE = re.compile(r"([\u0600-\u06ff])\1{2,}")
_REPEAT_LATIN_RE = re.compile(r"([A-Za-z])\1{2,}")
_SPACE_RE = re.compile(r"\s+")
_ZWNJ_RE = re.compile(r"[ ]*\u200c[ \u200c]*")
_NONWORD_RE = re.compile(r"[\W_]+")


def normalize(text: object) -> str:
    if not isinstance(text, str):
        return ""

    s = unicodedata.normalize("NFKC", text)
    s = s.translate(_TRANSLATE)
    s = _EMOJI_RE.sub(" ", s)
    s = _REPEAT_FA_RE.sub(r"\1", s)
    s = _REPEAT_LATIN_RE.sub(r"\1\1", s)
    s = _SPACE_RE.sub(" ", s)
    s = _ZWNJ_RE.sub(ZWNJ, s)

    return s.strip().strip(ZWNJ).strip()


def to_search_text(text: object) -> str:
    s = normalize(text).replace(ZWNJ, " ")
    s = _NONWORD_RE.sub(" ", s)
    return s.lower().strip()


def tokenize(text: object) -> list[str]:
    return to_search_text(text).split()


def build_search_text(*parts: object, dedupe_tokens: bool = True) -> str:
    tokens: list[str] = []

    for part in parts:
        tokens.extend(tokenize(part))

    if not dedupe_tokens:
        return " ".join(tokens)

    return " ".join(dict.fromkeys(tokens))

## 6. Shared retrieval contract

This section is adapted from Ali's shared retrieval interface.

`Evidence` is the standard retrieved-item representation used across the system. For product retrieval, the `meta` field contains structured product facts such as brand, price, rating, category, and fake-product flag.

In [6]:
# Adapted from src/retrieval/base.py — Owner: Ali

from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass, field
from typing import Any, Literal

EvidenceKind = Literal["product", "comment"]


@dataclass(slots=True)
class Evidence:
    id: str
    kind: EvidenceKind
    text: str
    score: float
    product_id: str | None = None
    title: str | None = None
    meta: dict[str, Any] = field(default_factory=dict)

    def citation(self) -> str:
        prefix = "product" if self.kind == "product" else "comment"
        return f"[{prefix}:{self.id}]"

    def as_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(slots=True)
class RetrievalFilters:
    price_min: float | None = None
    price_max: float | None = None
    brands: list[str] | None = None
    cat1: list[str] | None = None
    sub_cat: list[str] | None = None
    min_rate: float | None = None
    min_rate_count: int | None = None
    exclude_fake: bool = False
    product_ids: list[str] | None = None

    def is_empty(self) -> bool:
        return all(
            value in (None, False, [], {})
            for value in asdict(self).values()
        )


class Retriever(ABC):
    @abstractmethod
    def retrieve(
        self,
        query: str,
        top_k: int = 10,
        filters: RetrievalFilters | None = None,
    ) -> list[Evidence]:
        ...

## 7. Product retrieval

This section is adapted from Ali's product retrievers.

### BM25

BM25 performs lexical retrieval using normalized query tokens and the precomputed sparse product matrix.

### Dense retrieval

Dense retrieval encodes the query with `intfloat/multilingual-e5-base` and searches the prebuilt FAISS index.

### Hybrid retrieval

Hybrid retrieval combines the two rankings with Reciprocal Rank Fusion:

```text
RRF score(d) = Σ_i weight_i / (k + rank_i(d))
```

The current shared configuration uses:

```text
RRF_K = 60
DENSE_WEIGHT = 0.7
FETCH_DEPTH = 50
```

In [7]:
# Adapted from:
# - src/retrieval/products.py — Owner: Ali
# - src/retrieval/hybrid.py   — Owner: Ali

import json
from functools import lru_cache

import numpy as np
import pandas as pd
from scipy import sparse

MODEL_ID = "intfloat/multilingual-e5-base"
QUERY_PREFIX = "query: "
IVF_NPROBE = 32
OVER_FETCH_STEPS = (10, 50, 200)

RRF_K = 60
DENSE_WEIGHT = 0.7
FETCH_DEPTH = 50

STOPWORDS = {
    "یه", "یک", "که", "را", "رو", "از", "به", "با", "برای", "در", "و", "تا",
    "این", "آن", "هم", "می", "خوام", "میخوام", "می‌خوام", "باشه", "باشد",
    "است", "بود", "شود", "کن", "کنید", "چند", "چه", "خیلی", "نباشه", "هست",
    "بهترین", "معرفی", "دنبال", "میگردم", "چیه", "داره", "دارم", "بده",
}


@lru_cache(maxsize=1)
def load_meta() -> pd.DataFrame:
    df = pd.read_parquet(INDEX_DIR / "products_meta_v1.parquet")
    df["product_id"] = df["product_id"].astype(str)
    return df


@lru_cache(maxsize=1)
def load_bm25() -> tuple[sparse.csc_matrix, dict[str, int]]:
    matrix = sparse.load_npz(
        INDEX_DIR / "products_bm25_v1.npz"
    ).tocsc()

    vocab = json.loads(
        (INDEX_DIR / "products_bm25_vocab_v1.json").read_text(
            encoding="utf-8"
        )
    )

    return matrix, vocab


@lru_cache(maxsize=1)
def load_faiss():
    import faiss

    index = faiss.read_index(
        str(
            INDEX_DIR
            / f"products_e5base_{INDEX_TYPE}_v1.faiss"
        )
    )

    if hasattr(index, "nprobe"):
        index.nprobe = IVF_NPROBE

    return index


@lru_cache(maxsize=1)
def load_encoder():
    import torch
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(
        MODEL_ID,
        device=device,
    )
    model.max_seq_length = 128

    print("Dense encoder device:", device)

    return model


def filter_mask(
    meta: pd.DataFrame,
    filters: RetrievalFilters | None,
) -> np.ndarray | None:

    if filters is None or filters.is_empty():
        return None

    mask = np.ones(len(meta), dtype=bool)

    if filters.price_min is not None:
        mask &= (
            meta["price"] >= filters.price_min
        ).to_numpy(na_value=False)

    if filters.price_max is not None:
        mask &= (
            meta["price"] <= filters.price_max
        ).to_numpy(na_value=False)

    if filters.brands:
        mask &= meta["brand"].isin(
            filters.brands
        ).to_numpy()

    if filters.cat1:
        mask &= meta["cat1"].isin(
            filters.cat1
        ).to_numpy()

    if filters.sub_cat:
        mask &= meta["sub_cat"].isin(
            filters.sub_cat
        ).to_numpy()

    if filters.min_rate is not None:
        mask &= (
            meta["rate"] >= filters.min_rate
        ).to_numpy(na_value=False)

    if filters.min_rate_count is not None:
        mask &= (
            meta["rate_count"] >= filters.min_rate_count
        ).to_numpy()

    if filters.exclude_fake:
        mask &= ~meta["is_fake"].to_numpy()

    if filters.product_ids:
        mask &= meta["product_id"].isin(
            filters.product_ids
        ).to_numpy()

    return mask


def to_evidence(
    meta: pd.DataFrame,
    positions: np.ndarray,
    scores: np.ndarray,
) -> list[Evidence]:

    rows = meta.iloc[positions]

    return [
        Evidence(
            id=row.product_id,
            kind="product",
            text=row.title,
            score=float(score),
            product_id=row.product_id,
            title=row.title,
            meta={
                "brand": row.brand,
                "price": (
                    None
                    if pd.isna(row.price)
                    else float(row.price)
                ),
                "rate": (
                    None
                    if pd.isna(row.rate)
                    else float(row.rate)
                ),
                "rate_count": int(row.rate_count),
                "cat1": row.cat1,
                "sub_cat": row.sub_cat,
                "is_fake": bool(row.is_fake),
            },
        )
        for row, score in zip(
            rows.itertuples(index=False),
            scores,
        )
    ]


def top_positions(
    scores: np.ndarray,
    mask: np.ndarray | None,
    top_k: int,
) -> tuple[np.ndarray, np.ndarray]:

    if mask is None:
        limit = min(top_k, len(scores))
    else:
        scores = np.where(mask, scores, -np.inf)
        limit = min(top_k, int(mask.sum()))

    if limit <= 0:
        return (
            np.array([], dtype=int),
            np.array([], dtype=float),
        )

    candidates = np.argpartition(
        -scores,
        limit - 1,
    )[:limit]

    order = candidates[
        np.argsort(-scores[candidates])
    ]

    return order, scores[order]


class BM25Retriever(Retriever):
    def retrieve(
        self,
        query: str,
        top_k: int = 10,
        filters: RetrievalFilters | None = None,
    ) -> list[Evidence]:

        matrix, vocab = load_bm25()
        meta = load_meta()

        normalized_terms = tokenize(query)
        terms = [
            term
            for term in normalized_terms
            if term not in STOPWORDS
        ]

        term_ids = [
            vocab[term]
            for term in (terms or normalized_terms)
            if term in vocab
        ]

        if not term_ids:
            return []

        scores = np.asarray(
            matrix[:, term_ids].sum(axis=1)
        ).ravel()

        positions, values = top_positions(
            scores,
            filter_mask(meta, filters),
            top_k,
        )

        return to_evidence(
            meta,
            positions,
            values,
        )


class DenseRetriever(Retriever):
    def encode_query(
        self,
        query: str,
    ) -> np.ndarray:

        model = load_encoder()

        vector = model.encode(
            [QUERY_PREFIX + query],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        return vector.astype("float32")

    def retrieve(
        self,
        query: str,
        top_k: int = 10,
        filters: RetrievalFilters | None = None,
    ) -> list[Evidence]:

        index = load_faiss()
        meta = load_meta()
        mask = filter_mask(meta, filters)

        vector = self.encode_query(query)

        for step in OVER_FETCH_STEPS:
            depth = min(
                top_k * step,
                index.ntotal,
            )

            distances, positions = index.search(
                vector,
                depth,
            )

            scores = 1.0 - distances / 2.0

            positions = positions[0]
            scores = scores[0]

            keep = positions >= 0

            if mask is not None:
                valid_positions = positions.copy()
                valid_positions[valid_positions < 0] = 0
                keep &= mask[valid_positions]

            if (
                keep.sum() >= top_k
                or depth >= index.ntotal
            ):
                kept_positions = positions[keep][:top_k]
                kept_scores = scores[keep][:top_k]

                return to_evidence(
                    meta,
                    kept_positions,
                    kept_scores,
                )

        return []


class HybridRetriever(Retriever):
    def __init__(
        self,
        rrf_k: int = RRF_K,
        dense_weight: float = DENSE_WEIGHT,
        fetch_depth: int = FETCH_DEPTH,
    ) -> None:

        self.rrf_k = rrf_k
        self.dense_weight = dense_weight
        self.fetch_depth = fetch_depth

        self.dense = DenseRetriever()
        self.sparse = BM25Retriever()

    def retrieve(
        self,
        query: str,
        top_k: int = 10,
        filters: RetrievalFilters | None = None,
    ) -> list[Evidence]:

        depth = max(
            self.fetch_depth,
            top_k,
        )

        runs = (
            (
                self.dense.retrieve(
                    query,
                    top_k=depth,
                    filters=filters,
                ),
                self.dense_weight,
            ),
            (
                self.sparse.retrieve(
                    query,
                    top_k=depth,
                    filters=filters,
                ),
                1 - self.dense_weight,
            ),
        )

        scores: dict[str, float] = {}
        items: dict[str, Evidence] = {}

        for evidence_list, weight in runs:
            for rank, evidence in enumerate(
                evidence_list,
                start=1,
            ):
                scores[evidence.id] = (
                    scores.get(evidence.id, 0.0)
                    + weight / (self.rrf_k + rank)
                )

                items.setdefault(
                    evidence.id,
                    evidence,
                )

        ranked = sorted(
            scores.items(),
            key=lambda item: -item[1],
        )[:top_k]

        fused: list[Evidence] = []

        for product_id, score in ranked:
            evidence = items[product_id]

            fused.append(
                Evidence(
                    id=evidence.id,
                    kind=evidence.kind,
                    text=evidence.text,
                    score=round(score, 6),
                    product_id=evidence.product_id,
                    title=evidence.title,
                    meta={
                        **evidence.meta,
                        "fusion": "rrf",
                    },
                )
            )

        return fused


def build_product_retriever(
    backend: str = BACKEND,
) -> Retriever:

    backend = backend.lower()

    if backend == "hybrid":
        return HybridRetriever()

    if backend == "dense":
        return DenseRetriever()

    if backend == "bm25":
        return BM25Retriever()

    raise ValueError(
        f"Unknown retrieval backend: {backend!r}"
    )


product_retriever = build_product_retriever()

print(
    "Product retriever:",
    type(product_retriever).__name__,
)

Product retriever: HybridRetriever


## 8. Rule-based product-query handling

This section is adapted from Benyamin's rule-based filter extractor and product-discovery chain.

The rule-based extractor handles explicit numeric price constraints and the `exclude_fake` flag without using an LLM.

Examples:

```text
"زیر 20 میلیون تومان"
→ price_max = 200,000,000 rial

"اصل"
→ exclude_fake = True
```

Vague preferences remain part of the search query.

In [8]:
# Adapted from:
# - src/chains/product_filters.py   — Owner: Benyamin
# - src/chains/product_discovery.py — Owner: Benyamin

_NUMBER = r"(?P<number>\d+(?:[.,]\d+)?)"
_SCALE = r"(?P<scale>میلیون|هزار)?"
_CURRENCY = r"(?P<currency>تومان|تومن|ریال)"

_MAX_PRICE_RE = re.compile(
    rf"(?:زیر|کمتر از|حداکثر|تا)\s*"
    rf"{_NUMBER}\s*{_SCALE}\s*{_CURRENCY}"
)

_MIN_PRICE_RE = re.compile(
    rf"(?:بالای|بیشتر از|حداقل)\s*"
    rf"{_NUMBER}\s*{_SCALE}\s*{_CURRENCY}"
)


def _price_to_rial(
    match: re.Match[str],
) -> float:

    number = float(
        match.group("number").replace(",", ".")
    )

    scale = match.group("scale")

    if scale == "هزار":
        number *= 1_000
    elif scale == "میلیون":
        number *= 1_000_000

    if match.group("currency") in {
        "تومان",
        "تومن",
    }:
        number *= 10

    return number


@dataclass(slots=True)
class ProductFilterPlan:
    search_query: str
    price_min_rial: float | None = None
    price_max_rial: float | None = None
    exclude_fake: bool = False

    def to_retrieval_filters(
        self,
    ) -> RetrievalFilters:

        return RetrievalFilters(
            price_min=self.price_min_rial,
            price_max=self.price_max_rial,
            exclude_fake=self.exclude_fake,
        )


class RuleBasedFilterExtractor:
    def extract(
        self,
        query: str,
    ) -> ProductFilterPlan:

        clean_query = normalize(query)

        if not clean_query:
            raise ValueError(
                "query cannot be empty"
            )

        max_match = _MAX_PRICE_RE.search(
            clean_query
        )

        min_match = _MIN_PRICE_RE.search(
            clean_query
        )

        search_query = _MAX_PRICE_RE.sub(
            " ",
            clean_query,
        )

        search_query = _MIN_PRICE_RE.sub(
            " ",
            search_query,
        )

        search_query = (
            normalize(search_query)
            or clean_query
        )

        return ProductFilterPlan(
            search_query=search_query,
            price_min_rial=(
                _price_to_rial(min_match)
                if min_match
                else None
            ),
            price_max_rial=(
                _price_to_rial(max_match)
                if max_match
                else None
            ),
            exclude_fake=bool(
                re.search(
                    r"(?:اصل|اورجینال|فیک نباش|غیراصل نباش)",
                    clean_query,
                )
            ),
        )


@dataclass(slots=True)
class ProductDiscoveryResult:
    user_query: str
    filter_plan: ProductFilterPlan
    products: list[Evidence]


class ProductDiscoveryChain:
    def __init__(
        self,
        retriever: Retriever,
        extractor: RuleBasedFilterExtractor,
    ) -> None:

        self.retriever = retriever
        self.extractor = extractor

    def run(
        self,
        user_query: str,
        top_k: int = 5,
    ) -> ProductDiscoveryResult:

        if not user_query.strip():
            raise ValueError(
                "user_query cannot be empty"
            )

        if top_k < 1:
            raise ValueError(
                "top_k must be at least 1"
            )

        plan = self.extractor.extract(
            user_query
        )

        products = self.retriever.retrieve(
            plan.search_query,
            top_k=top_k,
            filters=plan.to_retrieval_filters(),
        )

        return ProductDiscoveryResult(
            user_query=user_query,
            filter_plan=plan,
            products=products,
        )


discovery_chain = ProductDiscoveryChain(
    retriever=product_retriever,
    extractor=RuleBasedFilterExtractor(),
)

print("Product discovery chain is ready.")

Product discovery chain is ready.


## 9. Product comparison layer

This is the task-specific comparison layer.

For each requested product:

1. The product-discovery chain resolves the query to the best retrieved product.
2. The direct product metadata is converted into a `ProductFacts` object.
3. Duplicate resolutions are detected.
4. Missing products are recorded.
5. Review evidence remains explicitly unavailable.
6. Inference remains explicitly disabled.

The output therefore keeps the three conceptual layers separate:

```text
facts      → direct structured product data
evidence   → pending real review retrieval
inference  → disabled
```

In [9]:
@dataclass(frozen=True, slots=True)
class ProductFacts:
    requested_query: str
    product_id: str
    title: str
    brand: str | None
    price_rial: float | None
    rate: float | None
    rate_count: int | None
    cat1: str | None
    sub_cat: str | None
    is_fake: bool | None
    retrieval_score: float
    citation: str

    @classmethod
    def from_evidence(
        cls,
        requested_query: str,
        evidence: Evidence,
    ) -> "ProductFacts":

        return cls(
            requested_query=requested_query,
            product_id=(
                evidence.product_id
                or evidence.id
            ),
            title=(
                evidence.title
                or evidence.text
            ),
            brand=evidence.meta.get(
                "brand"
            ),
            price_rial=evidence.meta.get(
                "price"
            ),
            rate=evidence.meta.get(
                "rate"
            ),
            rate_count=evidence.meta.get(
                "rate_count"
            ),
            cat1=evidence.meta.get(
                "cat1"
            ),
            sub_cat=evidence.meta.get(
                "sub_cat"
            ),
            is_fake=evidence.meta.get(
                "is_fake"
            ),
            retrieval_score=evidence.score,
            citation=evidence.citation(),
        )

    def as_dict(
        self,
    ) -> dict[str, Any]:

        return asdict(self)


@dataclass(frozen=True, slots=True)
class ProductComparisonResult:
    requested_products: list[str]
    facts: list[ProductFacts]
    missing_products: list[str]
    warnings: list[str]

    def as_dict(
        self,
    ) -> dict[str, Any]:

        return {
            "requested_products":
                self.requested_products,

            "facts": [
                fact.as_dict()
                for fact in self.facts
            ],

            "evidence": {
                "available": False,
                "items": [],
                "reason": (
                    "Real comment retrieval "
                    "is not available in the "
                    "current pipeline."
                ),
            },

            "inference": None,

            "missing_products":
                self.missing_products,

            "warnings":
                self.warnings,
        }


class ProductComparisonChain:
    def __init__(
        self,
        discovery: ProductDiscoveryChain,
    ) -> None:

        self.discovery = discovery

    def run(
        self,
        product_queries: list[str],
    ) -> ProductComparisonResult:

        cleaned_queries = [
            normalize(query)
            for query in product_queries
            if normalize(query)
        ]

        if len(cleaned_queries) < 2:
            raise ValueError(
                "At least two product queries "
                "are required."
            )

        facts: list[ProductFacts] = []
        missing_products: list[str] = []
        warnings: list[str] = []

        seen_product_ids: set[str] = set()

        for query in cleaned_queries:
            result = self.discovery.run(
                query,
                top_k=1,
            )

            if not result.products:
                missing_products.append(
                    query
                )
                continue

            product = result.products[0]

            product_id = (
                product.product_id
                or product.id
            )

            if product_id in seen_product_ids:
                warnings.append(
                    f"{query!r} resolved to "
                    f"an already-selected product "
                    f"({product_id})."
                )
                continue

            seen_product_ids.add(
                product_id
            )

            facts.append(
                ProductFacts.from_evidence(
                    requested_query=query,
                    evidence=product,
                )
            )

        return ProductComparisonResult(
            requested_products=
                cleaned_queries,
            facts=facts,
            missing_products=
                missing_products,
            warnings=warnings,
        )


comparison_chain = ProductComparisonChain(
    discovery=discovery_chain
)

print("Product comparison chain is ready.")

Product comparison chain is ready.


## 10. Example comparison

The current interface accepts separate product queries. Entity extraction from a free-form multi-product sentence is intentionally outside the current scope.

In [10]:
PRODUCT_QUERIES = [
    "سامسونگ گلکسی A55",
    "شیائومی ردمی نوت 13",
]

comparison_result = comparison_chain.run(
    PRODUCT_QUERIES
)

comparison_result.as_dict()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Dense encoder device: cpu


{'requested_products': ['سامسونگ گلکسی A55', 'شیائومی ردمی نوت 13'],
 'facts': [{'requested_query': 'سامسونگ گلکسی A55',
   'product_id': '927103',
   'title': 'موچین سکی مدل SS-505',
   'brand': 'متفرقه',
   'price_rial': 550000.0,
   'rate': 80.0,
   'rate_count': 3,
   'cat1': 'برس\u200cها و تجهیزات آرایشی',
   'sub_cat': 'beauty',
   'is_fake': False,
   'retrieval_score': 0.011475,
   'citation': '[product:927103]'},
  {'requested_query': 'شیائومی ردمی نوت 13',
   'product_id': '6266757',
   'title': 'فلوت ریکوردر کد 13',
   'brand': 'متفرقه',
   'price_rial': 389500.0,
   'rate': 74.0,
   'rate_count': 85,
   'cat1': 'سازهای بادی',
   'sub_cat': 'book & stationary & art',
   'is_fake': False,
   'retrieval_score': 0.011475,
   'citation': '[product:6266757]'}],
 'evidence': {'available': False,
  'items': [],
  'reason': 'Real comment retrieval is not available in the current pipeline.'},
 'inference': None,
 'missing_products': [],
 'warnings': []}

## 11. Comparison table

This table contains direct product facts only. No review-derived claim and no model-generated inference is included.

In [11]:
facts_df = pd.DataFrame(
    [
        {
            "requested_query":
                fact.requested_query,

            "product_id":
                fact.product_id,

            "title":
                fact.title,

            "brand":
                fact.brand,

            "price_toman": (
                None
                if fact.price_rial is None
                else fact.price_rial / 10
            ),

            "rate_0_100":
                fact.rate,

            "rate_count":
                fact.rate_count,

            "category":
                fact.cat1,

            "sub_category":
                fact.sub_cat,

            "is_fake":
                fact.is_fake,

            "retrieval_score":
                fact.retrieval_score,

            "citation":
                fact.citation,
        }
        for fact
        in comparison_result.facts
    ]
)

display(facts_df)

if comparison_result.missing_products:
    print(
        "Missing products:",
        comparison_result.missing_products,
    )

if comparison_result.warnings:
    print("Warnings:")
    for warning in comparison_result.warnings:
        print("-", warning)

print(
    "\nEvidence available:",
    comparison_result.as_dict()[
        "evidence"
    ]["available"],
)

print(
    "Inference:",
    comparison_result.as_dict()[
        "inference"
    ],
)

,requested_query,product_id,title,brand,price_toman,rate_0_100,rate_count,category,sub_category,is_fake,retrieval_score,citation
0,سامسونگ گلکسی A55,927103,موچین سکی مدل SS-505,متفرقه,55000.0,80.0,3,برس‌ها و تجهیزات آرایشی,beauty,False,0.011475,[product:927103]
1,شیائومی ردمی نوت 13,6266757,فلوت ریکوردر کد 13,متفرقه,38950.0,74.0,85,سازهای بادی,book & stationary & art,False,0.011475,[product:6266757]



Evidence available: False
Inference: None


## 12. Retrieval sanity check

The top retrieval candidates are useful for separating retrieval errors from comparison-layer errors.

If the intended product is not ranked first, the issue is product resolution/retrieval rather than fact extraction.

In [12]:
for query in PRODUCT_QUERIES:
    print("=" * 100)
    print("QUERY:", query)

    debug_result = discovery_chain.run(
        query,
        top_k=5,
    )

    for rank, product in enumerate(
        debug_result.products,
        start=1,
    ):
        print(
            rank,
            "|",
            product.product_id,
            "|",
            product.title,
            "| score:",
            round(product.score, 6),
        )

QUERY: سامسونگ گلکسی A55
1 | 927103 | موچین سکی مدل SS-505 | score: 0.011475
2 | 5297696 | ذره بین مدل M50 | score: 0.01129
3 | 9833690 | جاکلیدی مدل اسم آهو 55465 | score: 0.011111
4 | 9591787 | جاکلیدی مدل اسم شیوا 5455 | score: 0.010937
5 | 9813484 | جاکلیدی مدل اسم آوا 5455 | score: 0.010769
QUERY: شیائومی ردمی نوت 13
1 | 6266757 | فلوت ریکوردر کد 13 | score: 0.011475
2 | 12643024 | رومیزی بافتنی کد M13 | score: 0.01129
3 | 3438363 | قیچی مدل ابری کد 13 | score: 0.011111
4 | 10828100 | جامدادی مدل GA13 | score: 0.010937
5 | 4635709 | دفترچه یادداشت کد13 | score: 0.010769


In [13]:
queries = [
    "سامسونگ گلکسی A55",
    "شیائومی ردمی نوت 13",
]

bm25_debug = BM25Retriever()
dense_debug = DenseRetriever()
hybrid_debug = HybridRetriever()

for query in queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)

    print("\n--- BM25 ---")
    for i, ev in enumerate(bm25_debug.retrieve(query, top_k=5), 1):
        print(i, ev.score, ev.product_id, ev.title)

    print("\n--- DENSE ---")
    for i, ev in enumerate(dense_debug.retrieve(query, top_k=5), 1):
        print(i, ev.score, ev.product_id, ev.title)

    print("\n--- HYBRID ---")
    for i, ev in enumerate(hybrid_debug.retrieve(query, top_k=5), 1):
        print(i, ev.score, ev.product_id, ev.title)


QUERY: سامسونگ گلکسی A55

--- BM25 ---
1 13.155426979064941 6970940 هودی پسرانه مدل JUST A55
2 12.739564895629883 7263349 هودی زنانه مدل دخترک A55
3 12.349188804626465 6504118 هودی دخترانه مدل قلب A55 رنگ قرمز
4 12.148021697998047 8144446 محافظ صفحه نمایش گلکسی مدل PMMA-01 مناسب برای ساعت هوشمند سامسونگ Galaxy Watch Active 40mm
5 11.982025146484375 10684178 سایه چشم مک فیکس شماره A55

--- DENSE ---
1 0.8374930024147034 927103 موچین سکی مدل SS-505
2 0.8332407474517822 5297696 ذره بین مدل M50
3 0.832445502281189 9833690 جاکلیدی مدل اسم آهو 55465
4 0.8317388296127319 9591787 جاکلیدی مدل اسم شیوا 5455
5 0.8317080736160278 9813484 جاکلیدی مدل اسم آوا 5455

--- HYBRID ---
1 0.011475 927103 موچین سکی مدل SS-505
2 0.01129 5297696 ذره بین مدل M50
3 0.011111 9833690 جاکلیدی مدل اسم آهو 55465
4 0.010937 9591787 جاکلیدی مدل اسم شیوا 5455
5 0.010769 9813484 جاکلیدی مدل اسم آوا 5455

QUERY: شیائومی ردمی نوت 13

--- BM25 ---
1 19.09792137145996 384944 سایه چشم نوت سری Luminoussilkmono شماره 13
2 19.

In [14]:
meta = load_meta()
faiss_index = load_faiss()
bm25_matrix, vocab = load_bm25()

print("metadata rows:", len(meta))
print("faiss vectors:", faiss_index.ntotal)
print("bm25 rows:", bm25_matrix.shape[0])

metadata rows: 948352
faiss vectors: 948352
bm25 rows: 948352


In [15]:
meta = load_meta()

checks = {
    "A55": r"A55",
    "Samsung": r"سامسونگ|Samsung",
    "Galaxy A55": r"(سامسونگ|Samsung|Galaxy|گلکسی).*A55|A55.*(سامسونگ|Samsung|Galaxy|گلکسی)",

    "Redmi Note 13": r"(ردمی|Redmi).*13|13.*(ردمی|Redmi)",
    "Xiaomi": r"شیائومی|Xiaomi",
}

for name, pattern in checks.items():
    matched = meta[
        meta["title"].fillna("").astype(str).str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    ]

    print("\n" + "=" * 100)
    print(name, "->", len(matched), "products")

    display(
        matched[
            [
                "product_id",
                "title",
                "brand",
                "cat1",
                "sub_cat",
            ]
        ].head(20)
    )


A55 -> 118 products


,product_id,title,brand,cat1,sub_cat
3226,10684178,سایه چشم مک فیکس شماره A55,مک فیکس,آرایش چشم,beauty
29304,11675439,عصا دستی طرح عقاب مدل EA55LK,متفرقه,ابزار توانمند سازی,beauty
29311,11121446,عصای دستی طرح عقاب مدل EA55BR,متفرقه,ابزار توانمند سازی,beauty
47731,2889112,ژل بازی کیدز مدل KA550,متفرقه,اسباب بازی,toys and kids
50450,4842249,عروسک طرح دختر روسی کد wa55 ارتفاع 19 سانتی متر,متفرقه,اسباب بازی,toys and kids
51661,10380507,ژل بازی دنیای سرگرمی های کمیاب مدل خلاق شو کد ...,دنیای سرگرمی های کمیاب,اسباب بازی,toys and kids
56761,2829189,ژل بازی فانی ژل بازی مدل KBA550 بسته 8 عددی,متفرقه,اسباب بازی,toys and kids
64072,2761053,خمیر ژل بازی فانی ژل بازی مدل KA550,متفرقه,اسباب بازی,toys and kids
70198,10351549,فیجت ضد استرس دنیای سرگرمی های کمیاب مدل پاپیت...,دنیای سرگرمی های کمیاب,اسباب بازی,toys and kids
72479,10291147,اسباب بازی کوکی دنیای سرگرمی های کمیاب مدل اسب...,دنیای سرگرمی های کمیاب,اسباب بازی,toys and kids



Samsung -> 1929 products


,product_id,title,brand,cat1,sub_cat
309056,5549287,تبلت سامسونگ مدل Galaxy Tab A7 Lite - T225 ظرف...,سامسونگ,تبلت,toys and kids
309066,8417299,تبلت سامسونگ مدل Galaxy Tab A8 10.5 SM-X205 ظر...,سامسونگ,تبلت,toys and kids
309067,3699809,تبلت سامسونگ مدل Galaxy Tab A7 10.4 SM-T505 ظر...,سامسونگ,تبلت,toys and kids
309068,11650994,تبلت سامسونگ مدل Galaxy Tab A8 X205 ظرفیت 32 گ...,سامسونگ,تبلت,toys and kids
309073,9936864,تبلت سامسونگ مدل Galaxy Tab S6 Lite 2022 ظرفیت...,سامسونگ,تبلت,toys and kids
309080,2853965,تبلت سامسونگ مدل Galaxy TAB S6 Lite ظرفیت 64 گ...,سامسونگ,تبلت,toys and kids
309083,8421550,تبلت سامسونگ مدل Galaxy Tab S8 Ultra ظرفیت 256...,سامسونگ,تبلت,toys and kids
309084,9669313,تبلت سامسونگ مدل Galaxy Tab S8 Ultra ظرفیت 128...,سامسونگ,تبلت,toys and kids
309092,11650836,تبلت سامسونگ مدل Galaxy Tab S8 5G SM-X706B ظرف...,سامسونگ,تبلت,toys and kids
369289,10129606,پرچم رومیزی جاویدان تندیس پرگاس مدل سامسونگ کد 2,جاویدان تندیس پرگاس,"دست بافته‌ها, رودوزی و محصولات پارچه ای و چرمی",book & stationary & art


/tmp/ipykernel_1139/348525280.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  meta["title"].fillna("").astype(str).str.contains(


...

/tmp/ipykernel_1139/348525280.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  meta["title"].fillna("").astype(str).str.contains(



Redmi Note 13 -> 0 products


,product_id,title,brand,cat1,sub_cat



Xiaomi -> 2947 products


,product_id,title,brand,cat1,sub_cat
60699,4150711,ساختنی شیائومی مدل Desert Racing car,شیائومی,اسباب بازی,toys and kids
61303,12323269,ساختنی شیائومی مدل Jupiter Dawn XJXL01IQI,شیائومی,اسباب بازی,toys and kids
62934,7653279,ماشین بازی کنترلی شیائومی مدل XMYKC01CM,شیائومی,اسباب بازی,toys and kids
75207,7591509,فیجت ضد استرس شیائومی مدل MI FIDGET CUBE,شیائومی,اسباب بازی,toys and kids
78056,7447749,فیجت ضد استرس شیائومی مدل ZJMH02IQI,شیائومی,اسباب بازی,toys and kids
87681,5055492,ماشین اصلاح موی سر و صورت شیائومی مدل Enchen,شیائومی,اصلاح موی سر,rural goods
87756,10021826,ماشین اصلاح موی سر و صورت شیائومی مدل LFQ03KL,شیائومی,اصلاح موی سر,rural goods
87775,10026223,ماشین اصلاح موی سر و صورت شیائومی Grooming Kit...,شیائومی,اصلاح موی سر,rural goods
88170,10955224,ماشین اصلاح موی صورت شیائومی مدل 5,شیائومی,اصلاح موی صورت,rural goods
88301,6940722,ماشین اصلاح موی صورت شیائومی مدل MIJIA S500 20...,شیائومی,اصلاح موی صورت,rural goods


In [16]:
_, vocab = load_bm25()

for query in [
    "سامسونگ گلکسی A55",
    "شیائومی ردمی نوت 13",
]:
    print("\nQUERY:", query)

    for token in tokenize(query):
        print(
            f"{token:15}",
            "IN VOCAB" if token in vocab else "NOT IN VOCAB",
        )


QUERY: سامسونگ گلکسی A55
سامسونگ         IN VOCAB
گلکسی           IN VOCAB
a55             IN VOCAB

QUERY: شیائومی ردمی نوت 13
شیائومی         IN VOCAB
ردمی            NOT IN VOCAB
نوت             IN VOCAB
13              IN VOCAB


## 13. Current limitations and next integration points

### Review evidence

The current shared retrieval code does not expose a real comment index, so review evidence is deliberately represented as unavailable rather than fabricated.

```python
evidence = {
    "available": False,
    "items": []
}
```

### Inference

Inference is deliberately disabled in this notebook.

```python
inference = None
```

The complete future pipeline is expected to be:

```text
Product facts
    +
Retrieved review evidence
    ↓
Grounded comparison context
    ↓
Inference layer
```

The current notebook stops before the inference layer.